# Treinamento EfficientNetB7 - Melhores Parâmetros

In [ ]:
import os
import gc
import json
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.utils import resample
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.metrics import Recall, Precision
from tensorflow.keras.applications import EfficientNetB7
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from sklearn.metrics import classification_report, confusion_matrix, precision_score

# --- CONFIGURAÇÕES --
PASTAS_DATASET = [
    'imagens_acrais_benignas',
    'imagens_acrais_maligno'
]

CAMINHO_MODELOS = 'Modelos_B7'
os.makedirs(CAMINHO_MODELOS, exist_ok=True)


## Carregamento de Dados

In [ ]:
# CARREGAR E COMBINAR METADADOS ---
dfs = []
for pasta in PASTAS_DATASET:
    caminho_csv = os.path.join(pasta, 'metadata.csv')
    if os.path.exists(caminho_csv):
        df_temp = pd.read_csv(caminho_csv)
        df_temp['caminho_imagem'] = df_temp['isic_id'].apply(lambda x: os.path.join(pasta, f"{x}.jpg"))
        dfs.append(df_temp)
    else:
        print(f"Aviso: {caminho_csv} não encontrado.")

df_completo = pd.concat(dfs, ignore_index=True)

df_unico = df_completo.drop_duplicates(subset='isic_id')
df_existente = df_unico[df_unico['caminho_imagem'].apply(os.path.exists)].copy()

df_benigno = df_existente[df_existente['diagnosis_1'] == 'Benign']
df_maligno = df_existente[df_existente['diagnosis_1'] == 'Malignant']

df_filtrado = pd.concat([df_benigno, df_maligno]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Dataset total criado com {len(df_filtrado)} imagens (Benignas: {len(df_benigno)}, Malignas: {len(df_maligno)}).")


## Conjunto ISIC-images

In [ ]:
# === PREPARAÇÃO DO DATASET DE TESTE: ISIC-images ===
print("=== PREPARANDO DATASET DE TESTE: ISIC-images ===\n")

PASTA_ISIC = 'ISIC-images'

df_isic = pd.read_csv(os.path.join(PASTA_ISIC, 'metadata.csv'))

df_isic['caminho_imagem'] = df_isic['isic_id'].apply(
    lambda x: os.path.join(PASTA_ISIC, f"{x}.jpg")
)

df_isic = df_isic.drop_duplicates(subset='isic_id')
df_isic = df_isic[df_isic['caminho_imagem'].apply(os.path.exists)].copy()
df_isic_filtrado = df_isic[df_isic['diagnosis_1'].isin(['Benign', 'Malignant'])].copy()
df_isic_filtrado = df_isic_filtrado.reset_index(drop=True)

n_total    = len(df_isic_filtrado)
n_benign   = len(df_isic_filtrado[df_isic_filtrado['diagnosis_1'] == 'Benign'])
n_malignant = len(df_isic_filtrado[df_isic_filtrado['diagnosis_1'] == 'Malignant'])

print(f"Dataset ISIC-images organizado com sucesso!")
print(f"  Total de imagens : {n_total}")
print(f"  Benignas         : {n_benign}")
print(f"  Malignas         : {n_malignant}")
print(f"\nDistribuição das classes:")
print(df_isic_filtrado['diagnosis_1'].value_counts())


## Data Augmentation

In [ ]:
print("=== DADOS: VALIDAÇÃO (20% do df_filtrado) + TREINO EQUILIBRADO (oversampling) ===")

TAMANHO_B7 = (224, 224)
BATCH_SIZE = 8

datagen_val = ImageDataGenerator(validation_split=0.2)
val_gen = datagen_val.flow_from_dataframe(
    dataframe=df_filtrado,
    x_col="caminho_imagem",
    y_col="diagnosis_1",
    target_size=TAMANHO_B7,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    subset="validation",
    shuffle=False,
)

df_malignant = df_filtrado[df_filtrado["diagnosis_1"] == "Malignant"]
df_benign = df_filtrado[df_filtrado["diagnosis_1"] == "Benign"]
maior_classe = max(len(df_malignant), len(df_benign))
df_benign_upsampled = resample(
    df_benign, replace=True, n_samples=maior_classe, random_state=42
)
df_malignant_upsampled = resample(
    df_malignant, replace=True, n_samples=maior_classe, random_state=42
)
df_equilibrado = pd.concat([df_malignant_upsampled, df_benign_upsampled]).sample(
    frac=1, random_state=42
).reset_index(drop=True)

print(f"Treino equilibrado: {len(df_equilibrado)} imagens")
print(df_equilibrado["diagnosis_1"].value_counts())

datagen_balanced = ImageDataGenerator(
    validation_split=0.2,
    rotation_range=45,       
    width_shift_range=0.1,   
    height_shift_range=0.1,  
    horizontal_flip=True,    
    vertical_flip=True       
)

train_gen_balanced = datagen_balanced.flow_from_dataframe(
    dataframe=df_equilibrado,
    x_col="caminho_imagem",
    y_col="diagnosis_1",
    target_size=TAMANHO_B7,
    batch_size=BATCH_SIZE,
    class_mode="binary",
)


## Modelo EfficientNetB7

In [ ]:
def descongelar_ultimos_n_blocos(base_model, n_blocos):
    base_model.trainable = False
    blocos_descongelar = set(range(7 - n_blocos + 1, 8)) 
    for layer in base_model.layers:
        deve_descongelar = (
            any(layer.name.startswith(f"block{b}") for b in blocos_descongelar)
            or layer.name.startswith("top_")
        )
        if deve_descongelar and not isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = True
        else:
            layer.trainable = False
    n_treinaveis = sum(int(np.prod(w.shape)) for w in base_model.trainable_weights)
    return n_treinaveis

lr = 1e-4
n_unf = 1
DROPOUT_UL = 0.4
nome_teste = f"B7_LR{lr}_UL{n_unf}_DO{DROPOUT_UL}"
nome_arquivo = os.path.join(CAMINHO_MODELOS, f"{nome_teste}.keras")

print("\n" + "=" * 60)
print(f"INICIANDO TESTE: {nome_teste}")
print(f"Learning Rate: {lr} | Blocos descongelados: {n_unf} | Dropout: {DROPOUT_UL}")
print("=" * 60)

tf.keras.backend.clear_session()
gc.collect()

base_model = EfficientNetB7(
    weights="imagenet",
    include_top=False,
    input_shape=(TAMANHO_B7[0], TAMANHO_B7[1], 3),
)
n_treinaveis = descongelar_ultimos_n_blocos(base_model, n_unf)
print(f"Parâmetros treináveis na base (sem BN): {n_treinaveis:,}")

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(DROPOUT_UL)(x)
saida = Dense(1, activation="sigmoid")(x)
modelo_cnn = Model(inputs=base_model.input, outputs=saida)

modelo_cnn.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.Recall(name="recall")],
)

callbacks_lista = [
    EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True, verbose=1),
    ModelCheckpoint(
        filepath=nome_arquivo,
        monitor="val_recall",
        mode="max",
        save_best_only=True,
        verbose=1,
    ),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1, min_lr=1e-6)
]

history = modelo_cnn.fit(
    train_gen_balanced,
    epochs=30,
    validation_data=val_gen,
    callbacks=callbacks_lista,
    verbose=1,
)


## Teste e Avaliação

In [ ]:
# --- Teste nos dois conjuntos ---
print("\n" + "=" * 60)
print("AVALIANDO O MODELO")
print("=" * 60)

datagen_teste = ImageDataGenerator()
extra_gen = datagen_teste.flow_from_dataframe(
    dataframe=df_isic_filtrado,
    x_col="caminho_imagem",
    y_col="diagnosis_1",
    target_size=TAMANHO_B7,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=False,
)

print("\n[1] Avaliação no conjunto de Validação Padrão (20%)")
loss_val, acc_val, rec_val = modelo_cnn.evaluate(val_gen, verbose=0)
print(f"Loss: {loss_val:.4f} | Acurácia: {acc_val:.4f} | Recall: {rec_val:.4f}")

print("\n[2] Avaliação no conjunto ISIC-images")
loss_isic, acc_isic, rec_isic = modelo_cnn.evaluate(extra_gen, verbose=0)
print(f"Loss: {loss_isic:.4f} | Acurácia: {acc_isic:.4f} | Recall: {rec_isic:.4f}")

# Previsões
extra_gen.reset()
preds_prob = modelo_cnn.predict(extra_gen, verbose=1)
preds_class = (preds_prob > 0.5).astype(int).flatten()
y_true = extra_gen.classes

print("\nClassification Report (ISIC-images):")
print(classification_report(y_true, preds_class, target_names=list(extra_gen.class_indices.keys())))

cm = confusion_matrix(y_true, preds_class)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=extra_gen.class_indices.keys(),
            yticklabels=extra_gen.class_indices.keys())
plt.title("Matriz de Confusão (ISIC-images)")
plt.ylabel("Verdadeiro")
plt.xlabel("Previsto")
plt.show()
